In [3]:
import json

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tabulate

import geopandas as gpd
import xarray as xr

In [4]:
# used to use a flexible data directory root in settings_data_locations.json
from typing import Any
def apply_root_json(obj, root) -> Any:
    if isinstance(obj, dict):
        return {k: apply_root_json(v, root) for k, v in obj.items()}
    if isinstance(obj, str) and obj.startswith("{data_root}/"):
        return root.rstrip("/") + "/" + obj[len("{data_root}/"):]
    return obj

In [5]:
# print current directory
current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

with open("settings_data_locations.json", "r") as f:
    data_files = json.load(f)
data_files = apply_root_json(data_files, data_files["data_root"])


Current working directory: c:\Users\roelfsemam\downscaling\Kaya_downscaling\downscaling


# Directory single database

In [5]:
dir_GADM_single = data_files["GADM"]["dir_GADM_single"]
path_GADM_single = Path(dir_GADM_single) / "gadm_410.gpkg"
print(dir_GADM_single)

F:/data_downscaling/GADM/single database


In [6]:
# List layers - requires pyogrio (the current geopandas default)
layers_info = gpd.list_layers(path_GADM_single)
print("\nLayers in file:")
print(layers_info)

layer_name = layers_info["name"].iloc[0]
gdf_GADM_single = gpd.read_file(path_GADM_single, layer=layer_name)

print("\nNumber of features:", len(gdf_GADM_single))
print("Columns:", gdf_GADM_single.columns.tolist())
print("Geometry type(s):", gdf_GADM_single.geom_type.unique())
print("CRS:", gdf_GADM_single.crs)
print("Bounding box:", gdf_GADM_single.total_bounds)
#print(gdf_GADM_single.head(10))
print(gdf_GADM_single.sample(10))
gdf_GADM_single.to_csv("data/check/GADM_single.csv", index=False)


Layers in file:
       name geometry_type
0  gadm_410  MultiPolygon

Number of features: 356508
Columns: ['UID', 'GID_0', 'NAME_0', 'VARNAME_0', 'GID_1', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'ISO_1', 'HASC_1', 'CC_1', 'TYPE_1', 'ENGTYPE_1', 'VALIDFR_1', 'GID_2', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'HASC_2', 'CC_2', 'TYPE_2', 'ENGTYPE_2', 'VALIDFR_2', 'GID_3', 'NAME_3', 'VARNAME_3', 'NL_NAME_3', 'HASC_3', 'CC_3', 'TYPE_3', 'ENGTYPE_3', 'VALIDFR_3', 'GID_4', 'NAME_4', 'VARNAME_4', 'CC_4', 'TYPE_4', 'ENGTYPE_4', 'VALIDFR_4', 'GID_5', 'NAME_5', 'CC_5', 'TYPE_5', 'ENGTYPE_5', 'GOVERNEDBY', 'SOVEREIGN', 'DISPUTEDBY', 'REGION', 'VARREGION', 'COUNTRY', 'CONTINENT', 'SUBCONT', 'geometry']
Geometry type(s): ['MultiPolygon']
CRS: EPSG:4326
Bounding box: [-180.        -90.        180.         83.658333]
           UID GID_0       NAME_0 VARNAME_0     GID_1              NAME_1  \
280773  280774   RWA       Rwanda             RWA.2_1           Amajyepfo   
1060      1061   DZA      Algeria          

In [7]:
# save unique IDs per level with names to CSV file
out_dir = Path("data/check")

for level in ["0", "1", "2"]:
    cols = [f"GID_{level}", f"NAME_{level}"]
    unique_pairs = (gdf_GADM_single[cols]
                    .drop_duplicates()
                    .sort_values(cols)
                    .reset_index(drop=True))
    unique_pairs.to_csv(out_dir / f"GADM_GID_{level}.csv", index=False)
    print(f"GID_{level}: {len(unique_pairs):,} unique values -> GADM_GID_{level}.csv")

GID_0: 263 unique values -> GADM_GID_0.csv
GID_1: 3,669 unique values -> GADM_GID_1.csv
GID_2: 47,339 unique values -> GADM_GID_2.csv


In [8]:
# print first few rows
df_single = gdf_GADM_single.drop(columns="geometry")

# print selection
UIDs = ["NLD", "USA"]
gdf_GADM_selected_single = gdf_GADM_single[gdf_GADM_single["GID_0"].isin(UIDs)].drop(columns="geometry")
print(tabulate.tabulate(gdf_GADM_selected_single.sample(10), headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))
gdf_GADM_selected_single.to_csv("data/check/GADM_single_selected.csv", index=False)

+---------+---------+---------------+-------------+----------+------------+-------------+-------------+---------+----------+--------+----------+-------------+-------------+--------------+-----------------+-------------+-------------+----------+--------+----------+-------------+-------------+---------+----------+-------------+-------------+----------+--------+----------+-------------+-------------+---------+----------+-------------+--------+----------+-------------+-------------+---------+----------+--------+----------+-------------+--------------+---------------+--------------+----------+-------------+---------------+---------------+-----------+
|     UID | GID_0   | NAME_0        | VARNAME_0   | GID_1    | NAME_1     | VARNAME_1   | NL_NAME_1   | ISO_1   | HASC_1   | CC_1   | TYPE_1   | ENGTYPE_1   |   VALIDFR_1 | GID_2        | NAME_2          | VARNAME_2   | NL_NAME_2   | HASC_2   | CC_2   | TYPE_2   | ENGTYPE_2   | VALIDFR_2   | GID_3   | NAME_3   | VARNAME_3   | NL_NAME_3   | HASC

# Directory geopackage

ISO3 code and country name (263 countries)

In [6]:
dir_GADM_geopackage = data_files["GADM"]["dir_GADM_geopackage"]
path_GADM_geopackage = Path(dir_GADM_geopackage) / "gadm_410-levels.gpkg"
print(dir_GADM_geopackage)

F:/data_downscaling/GADM/geopackage


In [ ]:
# List layers - requires pyogrio (the current geopandas default)
layers_info = gpd.list_layers(path_GADM_geopackage)
print("\nLayers in file:")
print(layers_info)

#layer_name = layers_info["name"].iloc[0]
for layer_name in layers_info["name"]:
    gdf_GADM_geopackage_layer = gpd.read_file(path_GADM_geopackage, layer=layer_name)
    print(f"\n{'=' * 60}\nLayer: {layer_name}\n{'=' * 60}")
    print("\nNumber of features:", len(gdf_GADM_geopackage_layer))
    print("\nColumns:", gdf_GADM_geopackage_layer.columns.tolist())
    print("\nGeometry type(s):", gdf_GADM_geopackage_layer.geom_type.unique())
    print("\nCRS:", gdf_GADM_geopackage_layer.crs)
    print("\nBounding box:", gdf_GADM_geopackage_layer.total_bounds)
    #print(gdf_GADM_geopackage.head(10))
    gdf_geopackage_layer_selected = gdf_GADM_geopackage_layer.sample(5).drop(columns="geometry")
    print(tabulate.tabulate(gdf_geopackage_layer_selected, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))


Layers in file:
    name geometry_type
0  ADM_0  MultiPolygon
1  ADM_1  MultiPolygon
2  ADM_2  MultiPolygon
3  ADM_3  MultiPolygon
4  ADM_4  MultiPolygon
5  ADM_5  MultiPolygon

Layer: ADM_0

Number of features: 263

Columns: ['GID_0', 'COUNTRY', 'geometry']

Geometry type(s): ['MultiPolygon']

CRS: EPSG:4326

Bounding box: [-180.        -90.        180.         83.658333]
+---------+-----------+
| GID_0   | COUNTRY   |
+=========+===========+
| ZMB     | Zambia    |
+---------+-----------+
| URY     | Uruguay   |
+---------+-----------+
| CUB     | Cuba      |
+---------+-----------+
| SRB     | Serbia    |
+---------+-----------+
| TCD     | Chad      |
+---------+-----------+

Layer: ADM_1

Number of features: 3662

Columns: ['GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry']

Geometry type(s): ['MultiPolygon']

CRS: EPSG:4326

Bounding box: [-180.          -55.98000169  180.           83.658333  ]
+-------

: 

: 

In [ ]:
# create dataframe with two columns, first is level (0-5), second is description
df_levels = pd.DataFrame({
    "level": [0, 1, 2, 3, 4, 5],
    "description": ["National", "State/province/equivalent", "County/district/equivalent", "Commune/municipality/equivalent", "Available for 20 countries", "Available for France and Rwanda"]
})
df_levels.dtypes

In [ ]:
figure_dir = Path("../figures")

# Country outlines as basemap - ADM_0 is small (263 features), load once
gdf_basemap = gpd.read_file(path_GADM_geopackage, layer="ADM_0")

for i, layer_name in enumerate(layers_info["name"]):
    gdf_layer = gpd.read_file(path_GADM_geopackage, layer=layer_name)

    fig, ax = plt.subplots(figsize=(16, 8))
    gdf_basemap.plot(ax=ax, facecolor="white", edgecolor="black", linewidth=0.3)
    gdf_layer.plot(ax=ax, facecolor="lightblue", edgecolor="black")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_title(f"GADM {layer_name} ({len(gdf_layer)} features)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    level_value = int(layer_name.split("_")[-1])
    description = df_levels.loc[df_levels["level"] == level_value, "description"].iat[0]
    ax.set_title(f"{i}: GADM {layer_name}, {description} ({len(gdf_GADM_geopackage_layer)} features)")
    fig.savefig(figure_dir / f"GADM_{layer_name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved GADM_{layer_name}.png")

In [14]:
# check
dir_GADM_geopackage = data_files["GADM"]["dir_GADM_geopackage"]
path_GADM_geopackage = Path(dir_GADM_geopackage) / "gadm_410-levels.gpkg"

layer_name = "ADM_0"
gdf_GADM_geopackage_layer = gpd.read_file(path_GADM_geopackage, layer=layer_name)
check_Z = gdf_GADM_geopackage_layer[gdf_GADM_geopackage_layer["GID_0"].str.startswith("Z0")]  # check for Z0<number> codes
print(f"Z0 codes found: {check_Z['GID_0'].unique()}")
print(f"Country names for Z0 codes found: {check_Z['COUNTRY'].unique()}")
print(f"\n{'=' * 60}\nLayer: {layer_name}\n{'=' * 60}")
print("\nNumber of features:", len(gdf_GADM_geopackage_layer))
print("\nColumns:", gdf_GADM_geopackage_layer.columns.tolist())
print("\nGeometry type(s):", gdf_GADM_geopackage_layer.geom_type.unique())
print("\nCRS:", gdf_GADM_geopackage_layer.crs)
print("\nBounding box:", gdf_GADM_geopackage_layer.total_bounds)
#print(gdf_GADM_geopackage.head(10))
gdf_geopackage_layer_selected = gdf_GADM_geopackage_layer.sample(5).drop(columns="geometry")
print(tabulate.tabulate(gdf_geopackage_layer_selected, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.1f", intfmt=","))

Z0 codes found: ['Z02' 'Z03' 'Z08' 'Z01' 'Z04' 'Z05' 'Z07' 'Z09' 'Z06']
Country names for Z0 codes found: ['China' 'India' 'Pakistan']

Layer: ADM_0

Number of features: 263

Columns: ['GID_0', 'COUNTRY', 'geometry']

Geometry type(s): ['MultiPolygon']

CRS: EPSG:4326

Bounding box: [-180.        -90.        180.         83.658333]
+---------+-----------------------+
| GID_0   | COUNTRY               |
+=========+=======================+
| TKM     | Turkmenistan          |
+---------+-----------------------+
| CMR     | Cameroon              |
+---------+-----------------------+
| VIR     | Virgin Islands, U.S.  |
+---------+-----------------------+
| TJK     | Tajikistan            |
+---------+-----------------------+
| COG     | Republic of the Congo |
+---------+-----------------------+
